# Validate First Culture Organism

Converted from `sql/analysis/08_validate_first_culture_organism.sql`.

Validate first eligible culture-organism selection

One culture is selected per patient, standardized organism,
cohort, and source release before antibiotic testing or
susceptibility results are considered.

Only resolved standardized culture-organism records with a
usable shifted culture timestamp are eligible.

1. Final first-culture cohort sizes.

In [1]:
%sql
select
    cohort_id,
    organism_standardized,
    count(*) as first_culture_rows,
    count(distinct patient_key) as patients

from workspace.micro_dev.int_first_culture_organism

group by
    cohort_id,
    organism_standardized

order by
    cohort_id,
    first_culture_rows desc;

,cohort_id,organism_standardized,first_culture_rows,patients
0,blood,Escherichia coli,3021,3021
1,blood,Staphylococcus aureus,1731,1731
2,blood,Klebsiella pneumoniae,1050,1050
3,blood,Enterococcus faecalis,530,530
4,blood,Pseudomonas aeruginosa,384,384
5,blood,Enterococcus faecium,268,268
6,blood,Streptococcus agalactiae,262,262
7,blood,Enterobacter cloacae complex,243,243
8,blood,Proteus mirabilis,218,218
9,blood,Klebsiella oxytoca,212,212


2. Confirm intended grain.
Expected result: zero rows.

In [2]:
%sql
select
    source_version,
    cohort_id,
    patient_key,
    organism_standardized,
    count(*) as row_count

from workspace.micro_dev.int_first_culture_organism

group by
    source_version,
    cohort_id,
    patient_key,
    organism_standardized

having count(*) > 1;

,source_version,cohort_id,patient_key,organism_standardized,row_count


3. Confirm each selected culture is actually the earliest
eligible resolved culture for that patient / organism /
cohort.

Expected result: zero rows.

In [3]:
%sql
with expected as (

    select
        source_version,
        cohort_id,
        patient_key,
        organism_standardized,

        min(
            struct(
                culture_time_shifted,
                culture_key
            )
        ) as first_record

    from workspace.micro_dev.int_standardized_culture_organism

    where mapping_resolution_status = 'resolved'
      and culture_time_shifted is not null

    group by
        source_version,
        cohort_id,
        patient_key,
        organism_standardized

)

select
    f.source_version,
    f.cohort_id,
    f.patient_key,
    f.organism_standardized,

    f.culture_time_shifted as selected_time,
    f.culture_key as selected_culture_key,

    e.first_record.col1 as expected_time,
    e.first_record.col2 as expected_culture_key

from workspace.micro_dev.int_first_culture_organism f

inner join expected e
    on f.source_version = e.source_version
   and f.cohort_id = e.cohort_id
   and f.patient_key = e.patient_key
   and f.organism_standardized = e.organism_standardized

where f.culture_time_shifted <> e.first_record.col1
   or f.culture_key <> e.first_record.col2;

{"ts": "2026-10-08 22:53:43.664", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[FIELD_NOT_FOUND] No such struct field `col1` in `culture_time_shifted`, `culture_key`. SQLSTATE: 42704; line 49 pos 32\n\nJVM stacktrace:\norg.apache.spark.sql.AnalysisException\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.noSuchStructFieldInGivenFieldsError(QueryCompilationErrors.scala:4747)\n\tat org.apache.spark.sql.catalyst.expressions.ExtractValue$.findField(complexTypeExtractors.scala:262)\n\tat org.apache.spark.sql.catalyst.expressions.ExtractValue$.extractStructField$1(complexTypeExtractors.scala:108)\n\tat org.apache.spark.sql.catalyst.expressions.ExtractValue$.extractValue(complexTypeExtractors.scala:116)\n\tat org.apache.spark.sql.catalyst.expressions.ExtractValue$.apply(complexTypeExtractors.scala:57)\n\tat org.apache.spark.sql.catalyst.expressions.ExtractValue$.applyOrNull(complexTypeExtractors.scala:80)\n\tat org.apache.spark.sql.catalyst.expressions.package$Attrib

AnalysisException: [FIELD_NOT_FOUND] No such struct field `col1` in `culture_time_shifted`, `culture_key`. SQLSTATE: 42704; line 49 pos 32

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.noSuchStructFieldInGivenFieldsError(QueryCompilationErrors.scala:4747)
	at org.apache.spark.sql.catalyst.expressions.ExtractValue$.findField(complexTypeExtractors.scala:262)
	at org.apache.spark.sql.catalyst.expressions.ExtractValue$.extractStructField$1(complexTypeExtractors.scala:108)
	at org.apache.spark.sql.catalyst.expressions.ExtractValue$.extractValue(complexTypeExtractors.scala:116)
	at org.apache.spark.sql.catalyst.expressions.ExtractValue$.apply(complexTypeExtractors.scala:57)
	at org.apache.spark.sql.catalyst.expressions.ExtractValue$.applyOrNull(complexTypeExtractors.scala:80)
	at org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.$anonfun$resolveCandidates$2(package.scala:431)
	at scala.collection.LinearSeqOps.fold

4. Quantify resolved records excluded because the shifted
culture timestamp is missing.

In [ ]:
%sql
select
    cohort_id,
    organism_standardized,
    count(*) as missing_timestamp_rows,
    count(distinct patient_key) as affected_patients

from workspace.micro_dev.int_standardized_culture_organism

where mapping_resolution_status = 'resolved'
  and culture_time_shifted is null

group by
    cohort_id,
    organism_standardized

order by
    missing_timestamp_rows desc;

5. Reconcile final row count with distinct eligible
patient-organism combinations.

In [ ]:
%sql
with eligible as (

    select
        source_version,
        cohort_id,
        patient_key,
        organism_standardized

    from workspace.micro_dev.int_standardized_culture_organism

    where mapping_resolution_status = 'resolved'
      and culture_time_shifted is not null

    group by
        source_version,
        cohort_id,
        patient_key,
        organism_standardized

)

select
    (select count(*) from eligible)
        as eligible_patient_organism_groups,

    (
        select count(*)
        from workspace.micro_dev.int_first_culture_organism
    ) as selected_first_culture_rows;